In [1]:
import polyscope as ps
import numpy as np
import trimesh
import meshplot as mp 
from scipy.sparse.linalg import lsqr
import potpourri3d as pp3d
from scipy import sparse

from utils import get_index

### Uniform Laplacian Matrix
Recover coords from laplacian


In [2]:

### Define the pyramid and visualize
verts = np.array([[3, 0, 1],
                    [-1, -1, 0],
                    [1, -1, 0],
                    [1, 1, 0],
                    [-1, 1, 0]], dtype=np.float32)

faces = np.array([[0, 3, 2],
                       [0, 4, 3],
                       [0, 2, 1],
                       [0, 1, 4]], dtype=int)
mesh = trimesh.Trimesh(vertices=verts, faces=faces)
num_verts= verts.shape[0]
A = np.zeros((verts.shape[0], verts.shape[0]))
A[mesh.edges[:,0], mesh.edges[:,1]]= 1.
D = mesh.vertex_degree
I = np.eye(verts.shape[0])
L =  I- np.linalg.inv(D*I)@A

In [3]:
fixv = np.array([[0, 0, 0]])

p = mp.plot(verts, faces )
p.add_points((verts),shading={"point_size": 1})
p.add_points((fixv),c=np.array([[0,1,0]]),shading={"point_size": 1})

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(1.0, 0.0,…

2

In [4]:
# Recover vertices from Laplacian

# laplacian coordiante delta
delta = L@verts

# fix one vert
fix = np.zeros((1,num_verts))
fix[:,-1] = 1.
fixv = np.array([[0, 0, 0]])
#add the constraint to the system
L = np.concatenate((L, fix), axis=0)
delta = np.concatenate((delta, fixv), axis=0)

#solve for coordinates
updated_verts = np.zeros((num_verts,3))
for i in range(3):
    arr = lsqr(L, delta[:,i])[0] 
    updated_verts[:,i] = arr

In [5]:

p = mp.plot(updated_verts, faces )
p.add_points((verts),shading={"point_size": 1})
p.add_points((fixv),c=np.array([[0,1,0]]),shading={"point_size": 1})

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(1.9999999…

2

In [6]:
# Example mesh
path = '../data/meshes/bunny_to_edit.obj'
mesh = trimesh.load(path)
verts = mesh.vertices
faces = mesh.faces
num_verts = verts.shape[0]
vert_col = np.array(mesh.visual.vertex_colors).astype(np.float32)
vcol = vert_col[:, :3]/255.0

########### pick up anchor and handles from mesh vertex colors
anchor,handles  = get_index(vert_col,vert_col) 
num_anchor_verts = len(anchor)  

# define new handle positions
new_pos = np.array([[0.00,0.2,0.01]])


p = mp.plot(verts, faces, c=vcol)
p.add_points(new_pos,c=np.array([[0,1,0]]), shading={"point_size": 0.1})


/home/shaifali/miniconda3/envs/sa3df/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "float64". A coerced copy has been created.
  warnings.warn(
/home/shaifali/miniconda3/envs/sa3df/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float32" does not match required type "float64". A coerced copy has been created.
  warnings.warn(


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.016762…

1

### Color
- **Red:** Anchor (boundary condition), to be fixed.
- **Green:** Handle, to  edit

In [7]:
def get_lmatrix(L, anchor_indices,handles, num_verts):
    L= L.toarray()
    num_anchors = len(anchor_indices) 
    num_handles = len(handles)
    
    amatrix = np.zeros((num_anchors,num_verts))
    hmatrix = np.zeros((num_handles,num_verts))
    
    for i in range(num_anchors):
        amatrix[i, anchor_indices[i]] = 1.0
    for i in range(num_handles):
        hmatrix[i, handles[i]] = 1.0
    
    L = np.vstack((L,amatrix,hmatrix))     
    L = sparse.coo_matrix(L, shape=(L.shape)).tocsr()
    return L

In [8]:

def solve_laplacian(L, delta, num_verts):
    
    updated_verts = np.zeros((num_verts,3))
    for i in range(3):
        updated_verts[:, i] = lsqr(L, delta[:, i])[0]
        
    return updated_verts

In [9]:
# Laplacian matrix L
A = np.zeros((verts.shape[0], verts.shape[0]))
A[mesh.edges[:,0], mesh.edges[:,1]]= 1.
D = mesh.vertex_degree
I = np.eye(verts.shape[0])
L_dense =  I- np.linalg.inv(D*I)@A
L_dense = sparse.coo_matrix(L_dense, shape=(L_dense.shape)).tocsr()

# L' with new weights for anchors and handles
L_dense = get_lmatrix( L_dense, anchor,handles, num_verts)
# del' = L'.V
delta =  L_dense @ verts
# update delta to define new handle vertices
for i in range(len(handles)):
    delta[num_verts+num_anchor_verts+i, :] = new_pos[i] 

#solve for V' = L'^(-1) . del'
updated_verts = solve_laplacian(L_dense, delta, num_verts )



In [10]:

p = mp.plot(updated_verts, faces, c=vcol)
p.add_points(new_pos,c=np.array([[1,0,0]]), shading={"point_size": 0.1})

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.017858…

1

### Cotan Laplacian Matrix

In [11]:
# Laplacian matrix L
L_dense = pp3d.cotan_laplacian(verts, faces, denom_eps=1e-10)

# L' with new weights for anchors and handles
L_dense = get_lmatrix( L_dense, anchor,handles, num_verts)

# del' = L'.V
delta =  L_dense @ verts

# update delta to define new handle vertices
for i in range(len(handles)):
    delta[num_verts+num_anchor_verts+i, :] = new_pos[i] 

#solve for V' = L'^(-1) . del'
updated_verts = solve_laplacian(L_dense, delta, num_verts )



In [12]:

p = mp.plot(updated_verts, faces, c=vcol)
p.add_points(new_pos,c=np.array([[1,0,0]]), shading={"point_size": 0.1})

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.017689…

1